# Module 2 — From Direct LLM APIs to LangChain

## Classroom goal

Today we will learn how Python talks to an LLM through an API.

We will deliberately learn this in **two stages**.

### Stage 1 — Without LangChain

```text
Python → Gemini SDK → Gemini
Python → Groq SDK   → Groq model
```

We will notice that providers can have different:

- packages
- imports
- clients
- request structures
- response structures

### Stage 2 — With LangChain

```text
                 ┌→ Gemini
Python → LangChain
                 └→ Groq
```

We will use a more consistent calling style.

> **Naming reminder:** This lesson uses **Groq**, not **Grok**. Groq is the inference/API platform; Grok is associated with xAI.

### By the end of class, you should be able to

1. Load API keys from `.env`
2. Call Gemini directly
3. Call Groq directly
4. Explain why provider-specific integrations become repetitive
5. Connect Gemini and Groq through LangChain
6. Understand `invoke()`, `AIMessage`, and `.content`
7. Ask normal questions
8. Use a fixed/static prompt
9. Use one reusable prompt template
10. Start thinking about **Prompt Engineering**

# Part 0 — Before running the notebook

You should already have:

> **Compatibility note:** Use Python **3.10 or newer** for the current LangChain packages in this notebook. Python 3.12 is a good choice.

1. Installed Python
2. Installed VS Code
3. Installed the **Python** and **Jupyter** VS Code extensions
4. Created `.venv`
5. Activated `.venv`
6. Installed the project libraries:

```powershell
pip install -r requirements.txt
```

7. Added your real keys to `.env`
8. Selected the `.venv` kernel in VS Code

In [ ]:
# ==========================================================
# CELL 1: CHECK WHICH PYTHON IS RUNNING
# ==========================================================
#
# sys.executable = the full path of the Python interpreter
# sys.version    = the Python version
#
# We want the executable path to point to our project's
# .venv environment.
# ==========================================================

import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

If the path contains something similar to:

```text
...\your-project\.venv\Scripts\python.exe
```

the notebook is using the virtual environment correctly.

# Part 1 — Load API keys from `.env`

We do **not** want real API keys written directly inside Python code.

Our flow will be:

```text
.env
 ↓
python-dotenv
 ↓
os.getenv(...)
 ↓
Python variables
```

In [ ]:
# ==========================================================
# CELL 2: LOAD VALUES FROM .env
# ==========================================================

# os lets Python read environment variables.
import os

# load_dotenv reads the .env file in this project.
from dotenv import load_dotenv

load_dotenv()

# Read secret API keys.
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Read model names.
# The second argument is a fallback value if the variable
# is missing from .env.
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print real API keys in class.
print("Gemini key loaded:", bool(GEMINI_API_KEY))
print("Groq key loaded:", bool(GROQ_API_KEY))
print("Gemini model:", GEMINI_MODEL)
print("Groq model:", GROQ_MODEL)

In [ ]:
# ==========================================================
# CELL 3: SIMPLE SAFETY CHECK
# ==========================================================
#
# assert means: "this condition must be true".
#
# It lets us stop early with a friendly message if the
# student forgot to update .env.
# ==========================================================

assert GEMINI_API_KEY and GEMINI_API_KEY != "your_gemini_api_key_here", \
    "Add your real GEMINI_API_KEY to the .env file."

assert GROQ_API_KEY and GROQ_API_KEY != "your_groq_api_key_here", \
    "Add your real GROQ_API_KEY to the .env file."

print("Both API keys are ready.")

# Part 2 — Access Gemini DIRECTLY

First, we will **not use LangChain**.

We will use Google's official Python SDK.

```text
Our question
    ↓
Google Gemini SDK
    ↓
Gemini API
    ↓
Gemini model
    ↓
response.text
```

In [ ]:
# ==========================================================
# CELL 4: CREATE A GEMINI CLIENT
# ==========================================================

# Import Google's Gemini SDK.
from google import genai

# A client is an object our Python program uses to
# communicate with the Gemini API.
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client created.")

In [ ]:
# ==========================================================
# CELL 5: ASK GEMINI A NORMAL QUESTION
# ==========================================================

# A prompt can begin as a normal Python string.
question = "Explain what an API is to a 12-year-old student."

# generate_content(...) sends our input to the model.
#
# model=    tells Google which model to use.
# contents= is the text/input the model should read.
gemini_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=question,
)

# Google's SDK exposes the generated answer through .text
print(gemini_response.text)

## Understand the Gemini code

```python
genai.Client(...)
```

creates the Gemini API client.

```python
client.models.generate_content(...)
```

is the Google SDK method we used to generate a response.

```python
contents=question
```

contains our input.

```python
response.text
```

gives us the generated text.

Remember this provider-specific detail:

# Direct Gemini SDK → `response.text`

In [ ]:
# ==========================================================
# CELL 6: GEMINI PRACTICE — CHANGE ONLY THE QUESTION
# ==========================================================

question = "Why do programmers use virtual environments in Python?"

gemini_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=question,
)

print(gemini_response.text)

# Part 3 — Access Groq DIRECTLY

Now we will do the same conceptual job with another provider.

```text
Our question
    ↓
Groq Python SDK
    ↓
Groq API
    ↓
Model hosted on Groq
    ↓
response.choices[0].message.content
```

The goal is still **question → model → answer**, but the Python structure is different.

In [ ]:
# ==========================================================
# CELL 7: CREATE A GROQ CLIENT
# ==========================================================

# Import the official Groq Python client.
from groq import Groq

# Create the client and authenticate with our API key.
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq client created.")

In [ ]:
# ==========================================================
# CELL 8: ASK GROQ A NORMAL QUESTION
# ==========================================================

question = "Explain what an API is to a 12-year-old student."

# Groq's chat API expects a list of messages.
#
# Each message contains:
# role    = who is speaking
# content = what that person says
#
# "user" means the message comes from the human/user.
groq_response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role": "user",
            "content": question,
        }
    ],
)

# The direct Groq response structure is different from the
# direct Gemini response structure.
#
# choices[0] = first generated choice
# .message   = generated message object
# .content   = text inside that message
print(groq_response.choices[0].message.content)

# Compare the two DIRECT APIs

### Gemini

```python
from google import genai

client = genai.Client(api_key=...)
response = client.models.generate_content(
    model=...,
    contents=question
)
print(response.text)
```

### Groq

```python
from groq import Groq

client = Groq(api_key=...)
response = client.chat.completions.create(
    model=...,
    messages=[
        {"role": "user", "content": question}
    ]
)
print(response.choices[0].message.content)
```

## Same goal. Different provider interfaces.

| Job | Gemini SDK | Groq SDK |
|---|---|---|
| Import | `from google import genai` | `from groq import Groq` |
| Client | `genai.Client(...)` | `Groq(...)` |
| Generate | `models.generate_content()` | `chat.completions.create()` |
| Input | `contents=` | `messages=` |
| Read text | `response.text` | `response.choices[0].message.content` |

Imagine maintaining similar differences for 5, 10, or 20 providers.

# Part 4 — The limitation

There is nothing wrong with using provider SDKs directly. Direct SDKs can be valuable when you need provider-specific features.

But a multi-model application starts collecting different:

- imports
- client objects
- request formats
- response formats
- configuration styles
- provider-specific handling

Conceptually, our application wants something much simpler:

```text
choose model
   ↓
send question
   ↓
get answer
```

Can we give our application a **more common interface**?

# Enter LangChain

# Part 5 — What is LangChain?

LangChain is a framework for building applications around language models and related components.

For today's beginner class, remember one idea:

> **LangChain gives different chat-model integrations a more consistent application interface.**

### Analogy: universal travel adapter

Without an adapter:

```text
Device → Socket A
Device → Socket B
Device → Socket C
```

With a universal adapter:

```text
              ┌→ Socket A
Device → Adapter → Socket B
              └→ Socket C
```

For us:

```text
                ┌→ Gemini
Python → LangChain
                └→ Groq
```

LangChain does **not** make Gemini and Groq the same model. It makes our application code more consistent at the layer where we interact with them.

In [ ]:
# ==========================================================
# CELL 9: IMPORT LANGCHAIN MODEL INTEGRATIONS
# ==========================================================

# LangChain integration for Groq-hosted chat models.
from langchain_groq import ChatGroq

# LangChain integration for Google's Gemini chat models.
from langchain_google_genai import ChatGoogleGenerativeAI

print("LangChain integrations imported.")

In [ ]:
# ==========================================================
# CELL 10: CREATE BOTH LANGCHAIN MODEL OBJECTS
# ==========================================================
#
# We still create provider-specific model objects because
# Gemini and Groq are different services.
#
# The benefit appears after this setup: both support the
# LangChain .invoke(...) pattern.
# ==========================================================

groq_llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
)

gemini_llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
)

print("Groq LangChain model ready.")
print("Gemini LangChain model ready.")

# Part 6 — Understand `invoke()` and `.content`

## `invoke()`

```python
response = llm.invoke(question)
```

For today's lesson, read `invoke()` as:

> **Send this input through this LangChain component and return one result.**

## What comes back?

A LangChain chat model typically returns a message object such as an:

```text
AIMessage
```

It can contain the answer plus metadata.

## `.content`

For our text examples:

```python
response.content
```

gives us the generated answer.

Mental model:

```text
question
   ↓
llm.invoke(question)
   ↓
AIMessage
   ↓
.content
   ↓
generated text
```

In [ ]:
# ==========================================================
# CELL 11: ASK GROQ THROUGH LANGCHAIN
# ==========================================================

question = "Explain what an API is to a 12-year-old student."

groq_lc_response = groq_llm.invoke(question)

print(groq_lc_response.content)

In [ ]:
# ==========================================================
# CELL 12: ASK GEMINI THROUGH LANGCHAIN
# ==========================================================

question = "Explain what an API is to a 12-year-old student."

gemini_lc_response = gemini_llm.invoke(question)

print(gemini_lc_response.content)

# Stop and compare

## Direct SDKs

Gemini:

```python
response = gemini_client.models.generate_content(...)
print(response.text)
```

Groq:

```python
response = groq_client.chat.completions.create(...)
print(response.choices[0].message.content)
```

## LangChain

Gemini:

```python
response = gemini_llm.invoke(question)
print(response.content)
```

Groq:

```python
response = groq_llm.invoke(question)
print(response.content)
```

The underlying providers are different.

The **application-level calling pattern** is now much more consistent.

In [ ]:
# ==========================================================
# CELL 13: LOOK INSIDE AN AIMessage
# ==========================================================

response = groq_llm.invoke("Say hello in one short sentence.")

print("Python type:")
print(type(response))

print("\nWhole response object:")
print(response)

print("\nOnly the generated content:")
print(response.content)

For now, focus on:

```python
response.content
```

Later, the rest of the message object can be useful for things such as:

- token usage
- metadata
- debugging
- tool calls
- tracing

# Part 7 — Switch providers with one helper function

We can keep model objects in a dictionary:

```python
models = {
    "groq": groq_llm,
    "gemini": gemini_llm,
}
```

Then our higher-level code can choose a provider while still using:

```text
.invoke(...)
.content
```

In [ ]:
# ==========================================================
# CELL 14: ONE SIMPLE INTERFACE FOR BOTH MODELS
# ==========================================================

models = {
    "groq": groq_llm,
    "gemini": gemini_llm,
}


def ask_llm(provider, question):
    """Ask one of our LangChain chat models a question."""

    # Get the selected model object.
    llm = models[provider]

    # Both LangChain chat models support invoke().
    response = llm.invoke(question)

    # For this lesson, return the generated text.
    return response.content


print("Reusable ask_llm() function created.")

In [ ]:
# ==========================================================
# CELL 15: SAME FUNCTION — GROQ
# ==========================================================

answer = ask_llm(
    provider="groq",
    question="What is machine learning? Explain in 4 simple lines.",
)

print(answer)

In [ ]:
# ==========================================================
# CELL 16: SAME FUNCTION — GEMINI
# ==========================================================

answer = ask_llm(
    provider="gemini",
    question="What is machine learning? Explain in 4 simple lines.",
)

print(answer)

Only the provider changed:

```python
provider="groq"
```

versus:

```python
provider="gemini"
```

This is an example of **abstraction**: hiding lower-level differences behind a simpler common interface.

# Part 8 — Practice with NORMAL prompts

A normal prompt can simply be whatever the user types.

We will use Python's:

```python
input()
```

so students can ask different questions without editing the rest of the code.

In [ ]:
# ==========================================================
# CELL 17: INTERACTIVE QUESTION
# ==========================================================

provider = input("Choose provider (groq/gemini): ").strip().lower()
question = input("Ask your question: ")

if provider not in models:
    print("Please choose either 'groq' or 'gemini'.")
else:
    answer = ask_llm(provider, question)

    print("\n--- LLM ANSWER ---\n")
    print(answer)

## Student practice

Run the previous cell several times.

Try:

```text
What is an LLM?
```

```text
Explain tokens in simple words.
```

```text
What is the difference between AI and machine learning?
```

```text
Explain a Python function using a real-life analogy.
```

Ask the **same question** to both providers.

Do they answer exactly the same way? Why might different models respond differently even when the prompt is identical?

# Part 9 — Fixed / static prompts

So far, the user controlled the question.

Applications can also contain instructions written by the developer.

Example:

```text
You are a beginner-friendly AI teacher.
Explain what an API is.
Use simple English.
Use one everyday analogy.
Keep the answer under 150 words.
```

We can store this instruction in a Python variable and send the same prompt to either model.

In [ ]:
# ==========================================================
# CELL 18: CREATE A FIXED / STATIC PROMPT
# ==========================================================

STATIC_PROMPT = """
You are a beginner-friendly AI teacher.

Explain what an API is.

Rules:
- Use simple English.
- Assume the student has never programmed before.
- Give one everyday analogy.
- Keep the answer under 150 words.
"""

print(STATIC_PROMPT)

In [ ]:
# ==========================================================
# CELL 19: SAME STATIC PROMPT → GROQ
# ==========================================================

response = groq_llm.invoke(STATIC_PROMPT)
print(response.content)

In [ ]:
# ==========================================================
# CELL 20: SAME STATIC PROMPT → GEMINI
# ==========================================================

response = gemini_llm.invoke(STATIC_PROMPT)
print(response.content)

# Compare the responses

Ask yourself:

- Did both use an analogy?
- Did both stay beginner-friendly?
- Did both follow the word limit?
- What would happen if we removed the rules?
- What would happen if we changed the audience?

This is our first clue that **the wording and structure of the prompt matter**.

# Part 10 — One step toward reusable prompts

What if the teaching style stays the same but the **topic changes**?

Instead of copying the whole prompt repeatedly, we can create a template containing:

```text
{topic}
```

Then `{topic}` can become APIs, tokens, embeddings, transformers, Python functions, or anything else.

In [ ]:
# ==========================================================
# CELL 21: CREATE A CHAT PROMPT TEMPLATE
# ==========================================================

from langchain_core.prompts import ChatPromptTemplate

# {topic} is a variable / placeholder.
prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a beginner-friendly AI teacher.
            The student has no prior programming knowledge.
            Use simple English and practical analogies.
            """,
        ),
        (
            "human",
            """
            Explain {topic}.

            Requirements:
            - Keep it under 150 words.
            - Give one everyday analogy.
            - Give one simple example.
            """,
        ),
    ]
)

print("Prompt template created.")

## Why `system` and `human`?

### `system`

High-level behavior/instructions for the assistant.

Example:

```text
You are a beginner-friendly AI teacher.
```

### `human`

The user's actual request.

Example:

```text
Explain APIs.
```

We will study instruction design much more deeply in the next class.

# Part 11 — Build a simple LangChain chain

LangChain uses the pipe symbol:

```python
|
```

to compose compatible components.

```python
chain = prompt_template | groq_llm
```

Read it left to right:

```text
input variables
      ↓
prompt template
      ↓
     LLM
      ↓
   AIMessage
```

In [ ]:
# ==========================================================
# CELL 22: PROMPT TEMPLATE → GROQ
# ==========================================================

groq_chain = prompt_template | groq_llm

# The template expects a variable named {topic}, so invoke()
# receives a dictionary containing the value for "topic".
response = groq_chain.invoke(
    {
        "topic": "tokens in generative AI"
    }
)

print(response.content)

In [ ]:
# ==========================================================
# CELL 23: SAME PROMPT TEMPLATE → GEMINI
# ==========================================================

gemini_chain = prompt_template | gemini_llm

response = gemini_chain.invoke(
    {
        "topic": "tokens in generative AI"
    }
)

print(response.content)

Notice:

```python
prompt_template | groq_llm
```

and:

```python
prompt_template | gemini_llm
```

The prompt structure stayed the same. Only the model object changed.

In [ ]:
# ==========================================================
# CELL 24: REUSE THE SAME CHAIN WITH A NEW TOPIC
# ==========================================================

response = groq_chain.invoke(
    {
        "topic": "embeddings"
    }
)

print(response.content)

In [ ]:
# ==========================================================
# CELL 25: STUDENT PRACTICE
# ==========================================================
#
# Change ONLY the topic.
# Suggested topics:
# - transformers
# - context windows
# - API
# - machine learning
# - virtual environments
# - Python functions
# ==========================================================

my_topic = "transformers"

response = gemini_chain.invoke(
    {
        "topic": my_topic
    }
)

print(response.content)

# Part 12 — Final experiment: why does prompt wording matter?

Do **not** memorize prompt-engineering rules yet.

Just observe what happens when the **model stays the same** but the prompt changes.

In [ ]:
# ==========================================================
# CELL 26: VAGUE PROMPT
# ==========================================================

prompt_a = "Explain APIs."

response_a = groq_llm.invoke(prompt_a)

print(response_a.content)

In [ ]:
# ==========================================================
# CELL 27: MORE SPECIFIC PROMPT
# ==========================================================

prompt_b = """
You are teaching a 15-year-old student who has never programmed before.

Explain what an API is.

Requirements:
1. Use less than 150 words.
2. Start with a one-sentence definition.
3. Use a restaurant waiter analogy.
4. Give one software example.
5. Finish with one question to test the student's understanding.
"""

response_b = groq_llm.invoke(prompt_b)

print(response_b.content)

# Think before the next class

Compare `prompt_a` and `prompt_b`.

1. Which response was easier for a beginner to understand?
2. Which response had a more predictable structure?
3. Why did the model behave differently?
4. Which words in the prompt influenced the output?
5. Can we intentionally design prompts to get more useful and reliable responses?

Do not worry if you cannot answer all of these yet.

That is exactly where the next lesson begins:

# Next Class — Prompt Engineering

We will move from:

```text
"Ask the LLM something"
```

toward:

```text
"Design the instruction so the LLM understands
its role, task, context, constraints, examples,
and expected output."
```

# End-of-Class Challenge

Use the **same model** and the same topic:

```text
Machine Learning
```

Create three prompts.

### Prompt 1 — Very simple

```text
Explain machine learning.
```

### Prompt 2 — Add the audience

Tell the model who it is teaching.

### Prompt 3 — Add structure

Include:

- a role
- audience
- output length
- analogy requirement
- one example
- a requested output structure

Compare all three responses.

### Final question

> If the model stayed the same, but response quality changed when the prompt changed, what exactly are we programming?

Bring your answer to the next class.

# Quick Revision

## Direct Gemini

```python
response = gemini_client.models.generate_content(...)
print(response.text)
```

## Direct Groq

```python
response = groq_client.chat.completions.create(...)
print(response.choices[0].message.content)
```

## LangChain Gemini

```python
response = gemini_llm.invoke(question)
print(response.content)
```

## LangChain Groq

```python
response = groq_llm.invoke(question)
print(response.content)
```

## Reusable prompt chain

```python
chain = prompt_template | llm
response = chain.invoke({"topic": "APIs"})
print(response.content)
```

## The journey

```text
Different provider APIs
        ↓
Provider-specific integration differences
        ↓
LangChain abstraction
        ↓
Common invoke/content pattern
        ↓
Reusable prompt structures
        ↓
PROMPT ENGINEERING
```